In [1]:
import os
from PIL import Image
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as transforms

labels = pd.read_csv('../cifar-10/trainLabels.csv')
transform = transforms.ToTensor()

class_to_idx = {
    "airplane": 0,
    "automobile": 1,
    "bird": 2,
    "cat": 3,
    "deer": 4,
    "dog": 5,
    "frog": 6,
    "horse": 7,
    "ship": 8,
    "truck": 9
}

torch.manual_seed(42)

In [2]:
class CIFAR10Dataset(Dataset):
    def __init__(self, image_dir, labels):
        self.image_dir = image_dir
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image_id = self.labels.iloc[index]["id"]
        image_label = self.labels.iloc[index]["label"]

        image_path = os.path.join(self.image_dir, f"{image_id}.png")

        image = Image.open(image_path)
        image = transform(image)
        label = class_to_idx[image_label]

        return image, label

In [3]:
generator = torch.Generator().manual_seed(42)

train_dataset = CIFAR10Dataset(
    "../cifar-10/train",
    labels
)

train_size = int(0.8 * len(train_dataset))
val_size = int(0.1 * len(train_dataset))
test_size = int(0.1 * len(train_dataset))

train_dataset, val_dataset, test_dataset = random_split(
    train_dataset,
    [train_size, val_size, test_size],
    generator
)

print(f"Length of train dataset: {len(train_dataset)}")
print(f"Length of validation dataset: {len(val_dataset)}")
print(f"Length of test dataset: {len(test_dataset)}")

Length of train dataset: 40000
Length of validation dataset: 5000
Length of test dataset: 5000


In [4]:
image, label = train_dataset[0]
print("Train Dataset :-")
print(f"Image shape: {image.shape}")
print(f"Label: {label}")

image, label = val_dataset[0]
print("\nValidation Dataset :-")
print(f"Image shape: {image.shape}")
print(f"Label: {label}")

image, label = test_dataset[0]
print("\nTest Dataset :-")
print(f"Image shape: {image.shape}")
print(f"Label: {label}")

Train Dataset :-
Image shape: torch.Size([3, 32, 32])
Label: 6

Validation Dataset :-
Image shape: torch.Size([3, 32, 32])
Label: 7

Test Dataset :-
Image shape: torch.Size([3, 32, 32])
Label: 7


In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

images, labels_batch = next(iter(train_loader))
print("Train Dataloader :-")
print(f"Image Shape: {images.shape}")
print(f"Label: {labels_batch.shape}")

images, labels_batch = next(iter(val_loader))
print("\nValidation Dataloader :-")
print(f"Image Shape: {images.shape}")
print(f"Label: {labels_batch.shape}")

images, labels_batch = next(iter(test_loader))
print("\nTest Dataloader :-")
print(f"Image Shape: {images.shape}")
print(f"Label: {labels_batch.shape}")

Train Dataloader :-
Image Shape: torch.Size([32, 3, 32, 32])
Label: torch.Size([32])

Validation Dataloader :-
Image Shape: torch.Size([32, 3, 32, 32])
Label: torch.Size([32])

Test Dataloader :-
Image Shape: torch.Size([32, 3, 32, 32])
Label: torch.Size([32])


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [7]:
class CIFARBaseline(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            # Layer 1: 3*32*32 -> 32*32*32
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),

            # Layer 2: 32*32*32 -> 64*16*16
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Layer 3: 64*16*16 -> 128*8*8
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.network = nn.Sequential(
            nn.Flatten(),

            # Layer 1: 8192 -> 256
            nn.Linear(
                in_features=128*8*8,
                out_features=256
            ),
            nn.ReLU(),

            # Layer 2: 256 -> 128
            nn.Linear(
                in_features=256,
                out_features=128
            ),
            nn.ReLU(),

            # Layer 3: 128 -> 10
            nn.Linear(
                in_features=128,
                out_features=10
            )
        )

    def forward(self, x):
        x = self.cnn(x)
        x = self.network(x)

        return x

model = CIFARBaseline().to(device)
model

CIFARBaseline(
  (cnn): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=8192, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=0.001
)

epochs = 10
best_val_accuracy = 0
for epoch in range(epochs):

    # Training Loop
    model.train()
    running_train_loss = 0
    train_correct = 0
    train_total = 0

    for images, label_batch in train_loader:
        images, label_batch = images.to(device), label_batch.to(device)
        output = model(images)
        loss = loss_fn(output, label_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        predictions = output.argmax(dim=1)
        train_correct += (predictions == label_batch).sum().item()
        train_total += label_batch.size(0)

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = (train_correct / train_total) * 100

    # Validation Loop
    model.eval()
    running_val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.inference_mode():
        for images, label_batch in val_loader:
            images, label_batch = images.to(device), label_batch.to(device)
            output = model(images)
            loss = loss_fn(output, label_batch)

            running_val_loss += loss.item()

            predictions = output.argmax(dim=1)
            val_correct += (predictions == label_batch).sum().item()
            val_total += label_batch.size(0)

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = (val_correct / val_total) * 100

    print(f"\nEpoch {epoch + 1}:")
    print(f"Training Loss: {train_loss:.4f} | Training Acc: {train_accuracy:.2f}%")
    print(f"Validation Loss: {val_loss:.4f} | Validation Acc: {val_accuracy:.2f}%")
    
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(model.state_dict(), r'../models/CIFAR_Baseline.pt')
        print("Best Baseline Model Saved")


Epoch 1:
Training Loss: 1.4532 | Training Acc: 46.70%
Validation Loss: 1.0979 | Validation Acc: 61.54%
Best Baseline Model Saved

Epoch 2:
Training Loss: 0.9668 | Training Acc: 66.01%
Validation Loss: 0.8950 | Validation Acc: 68.18%
Best Baseline Model Saved

Epoch 3:
Training Loss: 0.7575 | Training Acc: 73.31%
Validation Loss: 0.8299 | Validation Acc: 71.38%
Best Baseline Model Saved

Epoch 4:
Training Loss: 0.6020 | Training Acc: 79.07%
Validation Loss: 0.7742 | Validation Acc: 73.68%
Best Baseline Model Saved

Epoch 5:
Training Loss: 0.4671 | Training Acc: 83.41%
Validation Loss: 0.8087 | Validation Acc: 73.52%

Epoch 6:
Training Loss: 0.3538 | Training Acc: 87.42%
Validation Loss: 0.8877 | Validation Acc: 73.68%

Epoch 7:
Training Loss: 0.2636 | Training Acc: 90.56%
Validation Loss: 0.9967 | Validation Acc: 72.70%

Epoch 8:
Training Loss: 0.1946 | Training Acc: 93.07%
Validation Loss: 1.0959 | Validation Acc: 73.38%

Epoch 9:
Training Loss: 0.1603 | Training Acc: 94.33%
Validatio